# Facial recognition
will create unique id for every face on will output all the photos the person is in

### Imprting everything needed

In [1]:
import pandas as pd 
import numpy as np
import cv2 
import os
import chromadb
from insightface.app import FaceAnalysis

/home/arc/tensuur/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = FaceAnalysis(name="buffalo_l", providers=['CUDAExecutionProvider'])
model.prepare(ctx_id=0, det_size=(640, 640))

/home/arc/tensuur/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/arc/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/arc/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/arc/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/arc/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /home/arc/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (64

In [3]:
chroma_client = chromadb.PersistentClient(path = "sdf")
collection = chroma_client.get_or_create_collection(name="face_embeddings",
                                                    metadata={"hnsw:space": "cosine"})

In [4]:
import uuid

In [7]:
dump = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled"
output = "/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/facial_output"
dis_thres = 0.4
os.makedirs(output, exist_ok=True)
for filename in os.listdir(dump):
    folder_path = os.path.join(dump, filename)
    if not os.path.isdir(folder_path):
        continue
    for img_path in os.listdir(folder_path):
        filepath = os.path.join(folder_path,img_path)
        print(filepath)
        img = cv2.imread(filepath)
        faces = model.get(img)
        for face in faces:
            embedding = face.embedding.tolist()
            results = collection.query(query_embeddings=[embedding], n_results=1)
            per_id = None
            if results['distances'][0] and results['distances'][0][0] < dis_thres:
                per_id = results['metadatas'][0][0]['per_id']
            else:
                per_id = f"Person_{str(uuid.uuid4())[:8]}"
                face_vector_id = str(uuid.uuid4())
                
                collection.add(
                    ids=[face_vector_id], 
                    embeddings=[embedding], 
                    metadatas=[{"per_id": per_id, "original_folder": folder_path,"filename": filename}]
                    )
            print(f"Image: {filepath}, Assigned ID: {per_id}")

/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled/Mark_Schweiker/Mark_Schweiker_0001.jpg
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled/Mark_Schweiker/Mark_Schweiker_0001.jpg, Assigned ID: Person_9f2d8637-768e-4b64-8c39-84ce823e977a
/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled/Mark_Schweiker/Mark_Schweiker_0002.jpg
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled/Mark_Schweiker/Mark_Schweiker_0002.jpg, Assigned ID: Person_9f2d8637-768e-4b64-8c39-84ce823e977a
/mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled/Paul_Celluci/Paul_Celluci_0001.jpg
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunneled/Paul_Celluci/Paul_Celluci_0001.jpg, Assigned ID: Person_9b4c9e6f
Image: /mnt/113117fa-e8af-4d8b-9b69-072b1b91c742/Code/facialRegonitionModel/lfw-deepfunnele

KeyboardInterrupt: 